In [1]:
import pandas as pd
import numpy as np
import pyarrow

df1 = pd.read_excel('../data/raw/online_retail_II.xlsx',sheet_name='Year 2009-2010')
df2 = pd.read_excel('../data/raw/online_retail_II.xlsx',sheet_name='Year 2010-2011')

df = pd.concat([df1, df2], ignore_index=True)

#df = pd.read_parquet('../data/processed/raw_combined.parquet')

print(f"Всего строк: {len(df)}")
print(f"Столбцы: {list(df.columns)}")

Всего строк: 1067371
Столбцы: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 78.8+ MB


In [3]:
def missing_report(df):
    missing_count = df.isna().sum()
    missing_pct = (missing_count / len(df) * 100).round(2)
    report = pd.DataFrame({'missing_count': missing_count, 
                           'missing_pct': missing_pct})
    return report[report['missing_count'] > 0].sort_values('missing_pct', ascending=False)

%time
missing_report(df)

CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 3.81 μs


,missing_count,missing_pct
Customer ID,243007,22.77
Description,4382,0.41


In [4]:
import os
print(os.getcwd())

/home/kamol/retail-sales/notebooks


In [5]:
df['Invoice'] = df['Invoice'].astype(str)

df['IsCancelled'] = df['Invoice'].str.startswith('C')
df['IsBadDebt'] = df['Invoice'].str.startswith('A')

print("Отменённые (C): ", df['IsCancelled'].sum())
print("Bad Debt (A): ", df['IsBadDebt'].sum())

Отменённые (C):  19494
Bad Debt (A):  6


In [6]:
df['InvoiceNumber'] = df['Invoice'].str.replace('[A-Za-z]', '', regex=True)
df[df['IsCancelled']][['Invoice','InvoiceNumber','IsCancelled']].head(5)

,Invoice,InvoiceNumber,IsCancelled
178,C489449,489449,True
179,C489449,489449,True
180,C489449,489449,True
181,C489449,489449,True
182,C489449,489449,True


In [7]:
has_letters = df['InvoiceNumber'].str.contains('[A-Za-z]', regex=True)
assert has_letters.sum() == 0, "В InvoiceNumber остались буквы после очистки!"

print("Проверка пройдена: InvoiceNumber содержит только цифры")

Проверка пройдена: InvoiceNumber содержит только цифры


In [8]:
matches_standard_format = df['StockCode'].astype(str).str.match(r'^\d{5}[A-Za-z]?$')

non_standard = df[~matches_standard_format]
print(f"Найденно нестандартных строк: {len(non_standard)}")

Найденно нестандартных строк: 7466


In [9]:
non_standard['StockCode'].value_counts()

StockCode
POST         2122
DOT          1446
M            1421
15056BL       923
C2            282
             ... 
DCGS0057        1
DCGS0073        1
DCGS0071        1
DCGS0066P       1
DCGS0067        1
Name: count, Length: 68, dtype: int64

In [10]:
non_standard_pattern = non_standard['StockCode'].astype(str).str.extract(r'([A-Za-z#]+)')
non_standard_pattern.value_counts()

0        
POST         2122
DOT          1446
M            1421
BL            923
C             283
LP            232
D             177
GR            123
DCGS          108
S             104
BANK          102
gift          100
bl             93
ADJUST         70
AMAZONFEE      43
DCGSSGIRL      25
DCGSSBOY       23
PADS           19
TEST           17
CRUK           16
B               6
m               5
SP              3
HC              1
GIFT            1
DCGSLBOY        1
DCGSLGIRL       1
J               1
Name: count, dtype: int64

In [11]:
known_service_codes = non_standard['StockCode'].value_counts().index.tolist()
print(known_service_codes)

['POST', 'DOT', 'M', '15056BL', 'C2', '79323LP', 'D', '79323GR', 'S', 'BANK CHARGES', '15056bl', 'ADJUST', 'AMAZONFEE', 'DCGS0058', 'gift_0001_20', 'gift_0001_30', 'DCGSSGIRL', 'DCGSSBOY', 'PADS', 'gift_0001_10', 'CRUK', 'DCGS0076', 'TEST001', 'DCGS0003', 'gift_0001_50', 'gift_0001_40', 'DCGS0069', 'B', 'DCGS0004', 'm', 'gift_0001_80', 'DCGS0072', 'DCGS0066N', 'DCGS0068', 'gift_0001_70', 'DCGS0070', 'ADJUST2', 'SP1002', 'TEST002', 'gift_0001_60', 'gift_0001_90', 'DCGS0062', 'DCGS0037', 'DCGS0044', '72024HC', 'DCGS0006', 'DCGS0016', 'DCGS0027', 'DCGS0036', 'DCGS0039', 'DCGS0060', 'DCGS0056', 'DCGS0059', 'GIFT', 'DCGSLBOY', 'DCGS0053', 'DCGSLGIRL', 'DCGS0075', 'DCGS0041', '47503J ', 'C3', 'DCGS0055', 'DCGS0074', 'DCGS0057', 'DCGS0073', 'DCGS0071', 'DCGS0066P', 'DCGS0067']


In [12]:
matches_standard_format = df['StockCode'].astype(str).str.match(r'^\d{5}[A-Za-z]?$')

df['IsServiceCode'] = ~matches_standard_format
print("Служебных строк: ", df['IsServiceCode'].sum())

Служебных строк:  7466


In [13]:
lookup_table = df[df['Description'].notna()].groupby('StockCode')['Description'].first()

print("Всего уникальных StockCode: ", len(lookup_table))

Всего уникальных StockCode:  4950


In [14]:
df['Description'] = df['Description'].fillna(df['StockCode'].map(lookup_table))
print("Осталось пропусков после фильтрации: ", df['Description'].isna().sum())

Осталось пропусков после фильтрации:  363


In [15]:
df['Description'] = df['Description'].fillna('Unknown')
df['HasUnknownDescription'] = df['Description'] == 'Unknown'

unknown_rows = df[df['HasUnknownDescription']]

print("Распределие по Quantity: (положительные и отрицательные): ", (unknown_rows['Quantity'] < 0).value_counts())
print()
print("IsCancelled среди строк: ", (unknown_rows['IsCancelled']).value_counts())
print()
print(unknown_rows[['Invoice','Quantity','Price','IsCancelled']])

Распределие по Quantity: (положительные и отрицательные):  Quantity
True     303
False     60
Name: count, dtype: int64

IsCancelled среди строк:  IsCancelled
False    363
Name: count, dtype: int64

       Invoice  Quantity  Price  IsCancelled
470     489521       -50    0.0        False
16186   490736      -200    0.0        False
19313   490976      -120    0.0        False
20975   491066       -67    0.0        False
22854   491167       -45    0.0        False
...        ...       ...    ...          ...
600756  542582        -6    0.0        False
634150  545566        -9    0.0        False
804713  561250        -2    0.0        False
804715  561255        -3    0.0        False
872310  567207         4    0.0        False

[363 rows x 4 columns]


In [16]:
df['IsStockAdjustment'] = df['HasUnknownDescription'] & (df['Price'] == 0)

print(df['IsStockAdjustment'].sum())

363


In [17]:
print(df['Customer ID'].isna().sum())
print(len(df) - df['Customer ID'].count())
print(df['Customer ID'].value_counts())

lookup_table1 = df['Customer ID'].nunique()
print("Всего уникальных ID: ", lookup_table1)

243007
243007
Customer ID
17841.0    13097
14911.0    11613
12748.0     7307
14606.0     6709
14096.0     5128
           ...  
14424.0        1
18184.0        1
13256.0        1
13017.0        1
15195.0        1
Name: count, Length: 5942, dtype: int64
Всего уникальных ID:  5942


In [18]:
df['IsGuest'] = df['Customer ID'].isna()

print("Всего гостевых заказов: ", df['IsGuest'].sum())

Всего гостевых заказов:  243007


In [37]:
countries = {"EIRE": "Ireland", "USA": "United States", "RSA": "South Africa"}
df['country_clean'] = df['Country'].replace(countries)
df['country_clean'].value_counts()

country_clean
United Kingdom          981330
Ireland                  17866
Germany                  17624
France                   14330
Netherlands               5140
Spain                     3811
Switzerland               3189
Belgium                   3123
Portugal                  2620
Australia                 1913
Channel Islands           1664
Italy                     1534
Norway                    1455
Sweden                    1364
Cyprus                    1176
Finland                   1049
Austria                    938
Denmark                    817
Unspecified                756
Greece                     663
Japan                      582
United States              535
Poland                     535
United Arab Emirates       500
Israel                     371
Hong Kong                  364
Singapore                  346
Malta                      299
Iceland                    253
Canada                     228
Lithuania                  189
South Africa             

In [38]:
territories = ['Unspecified', 'Channel Islands', 'Hong Kong', 'European Community', 'West Indies', 'Bermuda']
df['IsNotStandardGeography'] = df['country_clean'].isin(territories)
print("Всего заказов из нестандартных территорий: ", df['IsNotStandardGeography'].sum())

Всего заказов из нестандартных территорий:  2933


In [21]:
df['Revenue'] = df['Price']*df['Quantity']
df['Revenue'].value_counts()

only_valid = df[~df['IsServiceCode'] & 
    ~df['IsCancelled'] & 
    ~df['IsBadDebt']
    ]

print("Сумма всех строк: ", df['Revenue'].sum())
print("Сумма только валидных строк: ", only_valid['Revenue'].sum())

Сумма всех строк:  19287250.568
Сумма только валидных строк:  20047603.31


In [22]:
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['DayOfWeekNum'] = df['InvoiceDate'].dt.dayofweek
df['DayOfWeekStr'] = df['InvoiceDate'].dt.day_name()

In [39]:
flag_counts = ['IsCancelled', 'IsBadDebt', 'IsServiceCode', 'IsStockAdjustment', 'IsGuest', 'IsNotStandardGeography']
for col in flag_counts:
    assert df[col].isna().sum() == 0, f"В флаге {col} есть пропуски!"

assert df['InvoiceNumber'].str.contains('[A-Za-z]', regex=True).sum() == 0, "В InvoiceNumber есть пропуски!"

assert df['Revenue'].isna().sum() == 0, "В Revenue есть пропуски!"

assert df['Description'].isna().sum() == 0, "В Description есть пропуски!"

assert df['сountry_clean'].isna().sum() == 0, "В country_clean есть пропуски!"

assert df['сountry_clean'].isin(['EIRE', 'RSA', 'USA']).sum() == 0, "В country_clean остались старые значения!"

assert len(df) == 1067371, f"Ожидалось 1067371, получили {len(df)}!"

assert df['Year'].isin([2009, 2010, 2011]).sum() == len(df), "В Year есть лишние значения!"

print("Все проверки прошли успешно")

Все проверки прошли успешно


In [41]:
for col in df.columns:
    types_in_column = df[col].apply(type).nunique()
    if types_in_column > 1:
        print(f"{col}: обнаружено несколько типов: ", types_in_column)

In [42]:
df['StockCode'] = df['StockCode'].astype(str)
df['Description'] = df['Description'].astype(str)

In [43]:
df.to_parquet('../data/processed/filtered_combined.parquet', index=False)
print("Сохранено: ", df.shape)

Сохранено:  (1067371, 23)


In [44]:
import sqlite3

conn = sqlite3.connect('../data/processed/retail.db')

df.to_sql('sales', conn, if_exists='replace', index=False)

print("Данные загружены в SQLite")

Данные загружены в SQLite


In [45]:
#1
query_top_products ="""
                    SELECT Description, SUM(Revenue) as total_revenue
                    FROM sales
                    WHERE IsCancelled == 0 AND IsServiceCode == 0
                    GROUP BY Description
                    ORDER BY total_revenue DESC
                    LIMIT 10
                    """

result1 = pd.read_sql(query_top_products, conn)
result1

,Description,total_revenue
0,REGENCY CAKESTAND 3 TIER,344563.25
1,WHITE HANGING HEART T-LIGHT HOLDER,266923.55
2,"PAPER CRAFT , LITTLE BIRDIE",168469.60
3,JUMBO BAG RED RETROSPOT,150935.56
4,PARTY BUNTING,149187.05
5,ASSORTED COLOUR BIRD ORNAMENT,132187.92
6,PAPER CHAIN KIT 50'S CHRISTMAS,123141.54
7,CHILLI LIGHTS,85489.91
8,MEDIUM CERAMIC TOP STORAGE JAR,81700.92
9,ROTATING SILVER ANGELS T-LIGHT HLDR,74448.92


In [46]:
#2

month_year_revenue = """
                    SELECT Year, Month,  
                    ROUND(AVG(total_sales), 2) AS month_average_revenue,
                    ROUND(AVG(AVG(total_sales)) OVER(ORDER BY Year, Month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS months_3_average_revenue
                    FROM (SELECT Invoice, MAX(Year) AS Year, MAX(Month) AS Month, SUM(Revenue) as total_sales
                          FROM sales
                          WHERE IsCancelled = 0 AND IsServiceCode = 0
                          GROUP BY Invoice)
                    GROUP BY Year, Month
                    ORDER BY Year, Month
                    """

result2 = pd.read_sql(month_year_revenue, conn) 
result2

,Year,Month,month_average_revenue,months_3_average_revenue
0,2009,12,413.74,413.74
1,2010,1,478.13,445.94
2,2010,2,316.15,402.67
3,2010,3,394.51,396.26
4,2010,4,415.14,375.26
5,2010,5,323.28,377.64
6,2010,6,382.90,373.77
7,2010,7,383.91,363.36
8,2010,8,427.49,398.10
9,2010,9,445.13,418.84


In [47]:
#3
rank_countries = """
                SELECT country_clean, 
                CASE WHEN IsNotStandardGeography == 0 THEN 'Standard'
                ELSE 'Non-Standard' END AS country_type,
                ROUND(AVG(Revenue), 2) AS average_revenue, 
                ROUND(SUM(Revenue), 2) AS total_revenue,
                RANK() OVER(ORDER BY SUM(Revenue) DESC) AS revenue_rank
                FROM sales
                WHERE IsServiceCode == 0 AND IsCancelled == 0
                GROUP BY country_clean 
                ORDER BY total_revenue DESC
                """

result3 = pd.read_sql(rank_countries, conn)
result3

,country_clean,country_type,average_revenue,total_revenue,revenue_rank
0,United Kingdom,Standard,17.88,17179285.20,1
1,Ireland,Standard,36.65,625298.08,2
2,Netherlands,Standard,110.26,549952.66,3
3,Germany,Standard,24.18,387426.05,4
4,France,Standard,23.51,315336.35,5
5,Australia,Standard,93.18,168466.81,6
6,Spain,Standard,27.26,98612.79,7
7,Switzerland,Standard,30.58,94198.89,8
8,Sweden,Standard,67.10,86353.04,9
9,Denmark,Standard,88.46,68560.19,10


In [48]:
#4 
sum_comparision = """
                SELECT
                ROUND(SUM(CASE WHEN IsCancelled == 0 AND IsBadDebt == 0 THEN Revenue ELSE 0 END), 2) AS legit_sales,
                SUM(CASE WHEN IsCancelled == 1 THEN Revenue ELSE 0 END) AS cancelled_sales,
                SUM(CASE WHEN IsBadDebt == 1 THEN Revenue ELSE 0 END) AS bad_debt
                FROM sales
                """

result4 = pd.read_sql(sum_comparision, conn)
result4

,legit_sales,cancelled_sales,bad_debt
0,20961532.51,-1526667.86,-147614.08


In [55]:
#5
guest_regular_comparision = """
                            SELECT IsGuest, ROUND(AVG(order_total), 2) AS average_order_value, COUNT(*) AS total_orders
                            FROM (SELECT Invoice, MAX(IsGuest) AS IsGuest, SUM(Revenue) AS order_total
                                  FROM sales
                                  WHERE IsCancelled = 0 AND IsServiceCode = 0
                                  GROUP BY Invoice)
                            GROUP BY IsGuest
                            """

result5 = pd.read_sql(guest_regular_comparision, conn)
result5

,IsGuest,average_order_value,total_orders
0,0,474.82,36596
1,1,329.42,8109


In [50]:
#6
day_by_day_revenue = """
                    SELECT DayOfWeekStr, ROUND(AVG(Revenue), 2) AS average_revenue, SUM(Revenue) AS total_revenue
                    FROM sales
                    WHERE IsCancelled == 0 AND IsServiceCode == 0
                    GROUP BY DayOfWeekStr
                    ORDER BY average_revenue DESC
                    """

result6 = pd.read_sql(day_by_day_revenue, conn)
result6

,DayOfWeekStr,average_revenue,total_revenue
0,Saturday,24.25,9627.55
1,Friday,21.42,3206530.34
2,Thursday,20.81,4106407.62
3,Tuesday,20.70,3957669.07
4,Wednesday,19.24,3474851.71
5,Monday,18.76,3466988.26
6,Sunday,13.26,1825528.76


In [51]:
positive_reject = """
                SELECT *
                FROM sales
                WHERE Quantity > 0 AND Invoice LIKE "C%"
                """

result = pd.read_sql(positive_reject, conn)
result

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled,IsBadDebt,...,IsStockAdjustment,IsGuest,сountry_clean,IsNotStandardGeography,Revenue,Year,Month,DayOfWeekNum,DayOfWeekStr,country_clean
0,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,None,United Kingdom,1,0,...,0,1,United Kingdom,0,373.57,2010,2,0,Monday,United Kingdom


In [52]:
sum_revenue = """
              SELECT ROUND(SUM(Revenue), 2)
              FROM sales
              """

result11 = pd.read_sql(sum_revenue, conn)
result11

,"ROUND(SUM(Revenue), 2)"
0,19287250.57


In [53]:
#2.1
average_cost_of_item = """
                    SELECT Year, Month,  
                    ROUND(AVG(Revenue), 2) AS month_average_revenue,
                    ROUND(AVG(AVG(Revenue)) OVER(ORDER BY Year, Month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS months_3_average_revenue
                    FROM sales
                    GROUP BY Year, Month
                    ORDER BY Year, Month
                    """

result21 = pd.read_sql(average_cost_of_item, conn) 
result21

,Year,Month,month_average_revenue,months_3_average_revenue
0,2009,12,17.68,17.68
1,2010,1,19.78,18.73
2,2010,2,18.14,18.53
3,2010,3,18.45,18.79
4,2010,4,17.34,17.98
5,2010,5,17.42,17.74
6,2010,6,17.00,17.25
7,2010,7,17.23,17.22
8,2010,8,19.72,17.98
9,2010,9,20.28,19.08


In [54]:
top_products_df = pd.read_sql(query_top_products, conn)
month_revenue_df = pd.read_sql(month_year_revenue, conn) 
rank_countries_df = pd.read_sql(rank_countries, conn)
final_revenue_df = pd.read_sql(sum_comparision, conn)
users_comparison_df = pd.read_sql(guest_regular_comparision, conn)
days_revenue_df = pd.read_sql(day_by_day_revenue, conn)

with pd.ExcelWriter('../reports/summary_report.xlsx', engine='openpyxl') as writer:
    top_products_df.to_excel(writer, sheet_name='Top Products', index=False)
    month_revenue_df.to_excel(writer, sheet_name='Month Revenue', index=False)
    rank_countries_df.to_excel(writer, sheet_name='Rank Countries', index=False)
    final_revenue_df.to_excel(writer, sheet_name='Revenue', index=False)
    users_comparison_df.to_excel(writer, sheet_name='Users Comparison', index=False)
    days_revenue_df.to_excel(writer, sheet_name='Revenue by Days', index=False)